<a href="https://colab.research.google.com/github/salalisiasa-debug/-2026/blob/main/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 3. Потоковая обработка сенсорных событий в Spark Structured Streaming

**Дисциплина:** «Методы обработки больших данных»

· **Этап пайплайна:** `Stream → Process`

**Среда:** Google Colab, PySpark 4.2.0, встроенный источник `rate` (Kafka не требуется)

**Максимум:** 10 баллов

· **Лекция:** № 3 «Потоковые данные и распределённая обработка событий»

## Поля для заполнения

| Поле | Значение |
|---|---|
| **Фамилия, имя, отчество** | Салимуллина Алия Дамировна |
| **Группа** | ИНДОР-221|
| **Вариант** | 11 |
| **Дата выполнения** | 7.10.2026|

*(заполните таблицу и те же значения продублируйте в ячейке 1 — они попадут в отчёт)*

## Критерии оценки

| Часть | Содержание | Баллы |
|---|---|---:|
| 1 | Поток и телеметрия с `event_time`, опоздавшие события по варианту | 2 |
| 2 | Окна, агрегаты и корректный снимок результата | 2 |
| 3 | Эксперимент с watermark: два значения и измеримая метрика | 2 |
| 4 | Перенос на Kafka: что меняется в коде | 1 |
| 5 | Педагогическая часть: `TEACH CARD` и фрагмент занятия | 3 |

## Как работать

1. **Файл → Сохранить копию на Диске.** Работайте в своей копии.
2. В ячейке 1 укажите ФИО, группу и свой вариант. Параметры варианта считаются формулой —
   править их вручную нельзя.
3. Выполняйте ячейки сверху вниз. Места с пометкой `TODO` дописываете вы; до этого ячейка
   честно падает с `NotImplementedError`.
4. Поля вида **«Ваш ответ:»** и пустые ячейки таблиц заполняются текстом — они оцениваются
   так же, как код.
5. Перед сдачей запустите самопроверку в конце ноутбука.

> Все выводы должны опираться на **ваши** числа. Прогон `rate`-источника зависит от
> производительности среды, поэтому значения у всех разные — это нормально.

## Теоретический минимум

| Понятие | Коротко |
|---|---|
| `event_time` | когда событие физически произошло (ось анализа) |
| `processing_time` | когда движок его обработал |
| окно $W/S$ | интервал длиной $W$, повторяющийся через $S$; одно событие попадает примерно в $M \approx W/S$ окон |
| watermark $\Delta$ | $\mathrm{WM} = \max(t_{event}) - \Delta$; окна позади watermark финализируются, их состояние удаляется |
| состояние | «(робот, окно) → счётчики»; растёт с числом живых окон |

Компромисс, который вы измерите в части 3: **больше $\Delta$ — полнее данные, но больше
состояния и позже готов итог.**

Подробности — в лекции № 3 и в `README_lab_03.md`.

---
# Часть 1. Поток и телеметрия (2 балла)

## Ячейка 1. Вариант и параметры (дана готовой)

Параметры вычисляются формулой: группа `(n−1)//5` задаёт нагрузку, число роботов и частоту
опоздавших событий, позиция `(n−1)%5` — геометрию окна, сдвиг опоздавших событий,
дополнительную метрику и аудиторию для методической части. Заполните три строки вверху.

In [ ]:
# !pip -q install "pyspark==4.2.0"      # в Colab раскомментируйте эту строку
import json
import time
from typing import Dict, List

# =========== ЗАПОЛНИТЕ ===========
STUDENT_NAME  = "Салимуллина Алия Дамировна"      # <-- ваши ФИО
STUDENT_GROUP = "ИНДОР-221"                   # <-- ваша группа
VARIANT       = 11                          # <-- ваш вариант
# =================================
assert 1 <= VARIANT <= 25, "Вариант — целое число от 1 до 25"


def variant_params(n: int) -> dict:
    """Параметры варианта по таблице задания.

    Группа вариантов (n−1)//5 задаёт нагрузку и число роботов,
    позиция внутри группы (n−1)%5 — геометрию окна, долю и сдвиг опоздавших событий.
    """
    rows_per_sec = [20, 30, 40, 50, 60][(n - 1) // 5]
    robots       = [3, 4, 5, 6, 8][(n - 1) // 5]
    late_every   = [11, 17, 19, 23, 29][(n - 1) // 5]     # каждое k-е событие опаздывает
    window_s, slide_s = [(10, 5), (20, 5), (20, 10), (30, 10), (15, 5)][(n - 1) % 5]
    late_shift_s = [15, 20, 25, 30, 35][(n - 1) % 5]      # на сколько сдвинут event_time
    extra_metric = ["vibration_rms", "temp_spread", "temp_stddev",
                    "hot_share", "vibration_p95"][(n - 1) % 5]
    audience     = ["5–7 класс", "8–9 класс", "10–11 класс",
                    "кружок робототехники", "СПО"][(n - 1) % 5]
    return {
        "rows_per_sec": rows_per_sec,
        "robots": robots,
        "window_s": window_s,
        "slide_s": slide_s,
        "late_every": late_every,
        "late_shift_s": late_shift_s,
        "watermark_short": "10 seconds",
        "watermark_long": f"{late_shift_s + 10} seconds",   # заведомо больше сдвига
        "extra_metric": extra_metric,
        "audience": audience,
    }


CFG = variant_params(VARIANT)
RUN_SECONDS = 15        # длительность одного прогона потока

print(f"Студент : {STUDENT_NAME}, группа {STUDENT_GROUP}")
print(f"Вариант : {VARIANT}")
for key, value in CFG.items():
    print(f"  {key:<16}: {value}")
print(f"\nОкон на одно событие (теория): M ≈ W/S = "
      f"{CFG['window_s']}/{CFG['slide_s']} = {CFG['window_s'] / CFG['slide_s']:.0f}")
print(f"Ожидаемая нагрузка: {CFG['rows_per_sec']} событий/с, "
      f"доля опоздавших ≈ {100 / CFG['late_every']:.1f} %")



def todo(step):
    raise NotImplementedError(f"Не выполнен шаг TODO {step} — допишите код в отмеченном месте")

Студент : Салимуллина Алия Дамировна, группа ИНДОР-221
Вариант : 11
  rows_per_sec    : 40
  robots          : 5
  window_s        : 10
  slide_s         : 5
  late_every      : 19
  late_shift_s    : 15
  watermark_short : 10 seconds
  watermark_long  : 25 seconds
  extra_metric    : vibration_rms
  audience        : 5–7 класс

Окон на одно событие (теория): M ≈ W/S = 10/5 = 2
Ожидаемая нагрузка: 40 событий/с, доля опоздавших ≈ 5.3 %


## Ячейка 2. Сессия Spark

**TODO 2.** Создайте `SparkSession` в локальном режиме.

Подсказки: мастер `local[*]` (потоку нужно минимум два потока выполнения — один занимает
приёмник источника); `spark.sql.shuffle.partitions = 4`, иначе Spark создаст 200 почти пустых
партиций и каждый micro-batch будет длиться лишние секунды; `setLogLevel("ERROR")` уберёт шум
в выводе. В конце напечатайте версию Spark и число ядер.

In [ ]:
!pip -q install "pyspark==4.2.0"
from pyspark.sql import SparkSession, functions as F
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName(f"Lab03_StructuredStreaming_v{VARIANT}")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.streaming.metricsEnabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)
print("Ядер    :", spark.sparkContext.defaultParallelism)
print("Shuffle :", spark.conf.get("spark.sql.shuffle.partitions"))
# TODO 2: создайте сессию spark и напечатайте её параметры


Spark   : 4.2.0
Master  : local[*]
Ядер    : 2
Shuffle : 4


## Ячейка 3. Телеметрия робота

**TODO 3.** Напишите функцию `build_telemetry(source, cfg)`, которая превращает поток `rate`
(колонки `timestamp`, `value`) в телеметрию:

| Колонка | Как считается |
|---|---|
| `robot_id` | `"robot-" + (value % cfg["robots"])` |
| `is_late` | `value % cfg["late_every"] == 0` — «правда» эксперимента: какое событие опоздало |
| `event_time` | для опоздавших `timestamp − cfg["late_shift_s"] секунд`, иначе `timestamp` |
| `motor_temp` | `43.0 + (value % 25) * 0.55` |
| `vibration` | `abs(sin(value / 8))` |

Сначала проверьте логику на **конечной** таблице `spark.range(0, 12)` с колонкой
`timestamp = current_timestamp()`: в потоке искать ошибку в выражении гораздо дороже.
Напечатайте таблицу и число опоздавших событий в примере.

**Важно:** сдвигать нужно `event_time` в **прошлое**. Событие «из будущего» watermark не
отбросит, и эксперимент части 3 не получится.

In [ ]:
def build_telemetry(source, cfg: dict):
    """Превратить сырой поток rate в телеметрию роботов."""
    """Превратить сырой поток rate в телеметрию роботов.

    Логика одна и та же для потока и для статической таблицы, поэтому функция
    принимает DataFrame: так её можно отладить на конечных данных.
    """
    return (
        source
        .withColumn("robot_id", F.concat(F.lit("robot-"),
                                         (F.col("value") % cfg["robots"]).cast("string")))
        .withColumn("is_late", F.col("value") % cfg["late_every"] == 0)
        .withColumn("event_time",
                    F.when(F.col("is_late"),
                           F.col("timestamp") - F.expr(f"INTERVAL {cfg['late_shift_s']} SECONDS"))
                    .otherwise(F.col("timestamp")))
        .withColumn("motor_temp", F.lit(43.0) + (F.col("value") % 25).cast("double") * 0.55)
        .withColumn("vibration", F.abs(F.sin(F.col("value") / 8.0)))
        .select("robot_id", "event_time", "is_late", "motor_temp", "vibration")
    )


# Отладка логики на конечной таблице: те же выражения, но без потока.
preview_source = (spark.range(0, 12)
                  .withColumnRenamed("id", "value")
                  .withColumn("timestamp", F.current_timestamp()))
preview = build_telemetry(preview_source, CFG)
preview.show(12, truncate=False)
print("Опоздавших в примере:", preview.filter("is_late").count(), "из", preview.count())


+--------+--------------------------+-------+----------+-------------------+
|robot_id|event_time                |is_late|motor_temp|vibration          |
+--------+--------------------------+-------+----------+-------------------+
|robot-0 |2026-10-10 15:55:26.755138|true   |43.0      |0.0                |
|robot-1 |2026-10-10 15:55:41.755138|false  |43.55     |0.12467473338522769|
|robot-2 |2026-10-10 15:55:41.755138|false  |44.1      |0.24740395925452294|
|robot-3 |2026-10-10 15:55:41.755138|false  |44.65     |0.36627252908604757|
|robot-4 |2026-10-10 15:55:41.755138|false  |45.2      |0.479425538604203  |
|robot-0 |2026-10-10 15:55:41.755138|false  |45.75     |0.5850972729404622 |
|robot-1 |2026-10-10 15:55:41.755138|false  |46.3      |0.6816387600233341 |
|robot-2 |2026-10-10 15:55:41.755138|false  |46.85     |0.7675435022360271 |
|robot-3 |2026-10-10 15:55:41.755138|false  |47.4      |0.8414709848078965 |
|robot-4 |2026-10-10 15:55:41.755138|false  |47.95     |0.9022675940990952 |

---
# Часть 2. Окна, агрегаты и снимок (2 балла)

## Ячейка 4. Запуск потока с окнами

**TODO 4.** Напишите функцию `run_stream(cfg, watermark, seconds, name)`:

1. источник `rate` с `rowsPerSecond = cfg["rows_per_sec"]` и `numPartitions = 2`;
2. `build_telemetry`;
3. `withWatermark("event_time", watermark)`;
4. группировка по `window(event_time, "{W} seconds", "{S} seconds")` и `robot_id`;
5. агрегаты: `events` (счёт), `late_events` (сумма `is_late`), `avg_temp`, `max_temp` и
   дополнительная метрика вашего варианта (`cfg["extra_metric"]`);
6. запись в `memory`-приёмник, `outputMode("update")`, триггер 1 секунда;
7. ожидание `seconds` секунд, сохранение `list(query.recentProgress)`, затем `query.stop()`.

**Не используйте `query.processAllAvailable()`.** Источник `rate` бесконечен: «все доступные
данные» у него не заканчиваются, и вызов может заблокировать ячейку надолго. Останавливайте
поток по таймеру.

Функция возвращает `(таблица обновлений, список метрик батчей)`.

Выражения дополнительных метрик:

```python
AGGREGATES = {
    "vibration_rms": F.sqrt(F.avg(F.pow("vibration", 2))).alias("extra_metric"),
    "temp_spread":   (F.max("motor_temp") - F.min("motor_temp")).alias("extra_metric"),
    "temp_stddev":   F.stddev_samp("motor_temp").alias("extra_metric"),
    "hot_share":     F.avg((F.col("motor_temp") > 50).cast("double")).alias("extra_metric"),
    "vibration_p95": F.percentile_approx("vibration", 0.95).alias("extra_metric"),
}
```

In [ ]:
AGGREGATES = {
    "vibration_rms": F.sqrt(F.avg(F.pow("vibration", 2))).alias("extra_metric"),
    "temp_spread":   (F.max("motor_temp") - F.min("motor_temp")).alias("extra_metric"),
    "temp_stddev":   F.stddev_samp("motor_temp").alias("extra_metric"),
    "hot_share":     F.avg((F.col("motor_temp") > 50).cast("double")).alias("extra_metric"),
    "vibration_p95": F.percentile_approx("vibration", 0.95).alias("extra_metric"),
}


def run_stream(cfg: dict, watermark: str, seconds: int = RUN_SECONDS, name: str = "robot_windows"):
    """Запустить поток на заданное время и вернуть (снимок результата, метрики прогресса).

    Остановка по таймеру, а не processAllAvailable(): источник rate порождает данные
    непрерывно, и «дождаться всех доступных данных» у него нельзя — метод может
    блокировать выполнение надолго.
    """
    source = (spark.readStream.format("rate")
              .option("rowsPerSecond", cfg["rows_per_sec"])
              .option("numPartitions", 2)
              .load())
    telemetry = build_telemetry(source, cfg)
    windowed = (
        telemetry
        .withWatermark("event_time", watermark)
        .groupBy(F.window("event_time", f"{cfg['window_s']} seconds",
                          f"{cfg['slide_s']} seconds"), "robot_id")
        .agg(F.count("*").alias("events"),
             F.sum(F.col("is_late").cast("int")).alias("late_events"),
             F.avg("motor_temp").alias("avg_temp"),
             F.max("motor_temp").alias("max_temp"),
             AGGREGATES[cfg["extra_metric"]])
    )
    query = (windowed.writeStream
             .format("memory").queryName(name)
             .outputMode("update")
             .trigger(processingTime="1 second")
             .start())
    started = time.time()
    while time.time() - started < seconds:
        time.sleep(1)
    progress = list(query.recentProgress)
    query.stop()
    return spark.sql(f"SELECT * FROM {name}"), progress


raw_updates, progress_short = run_stream(CFG, CFG["watermark_short"], name="w_short")
print("Строк в memory-sink:", raw_updates.count())
print("Уникальных пар (окно, робот):", raw_updates.select("window", "robot_id").distinct().count())

Строк в memory-sink: 175
Уникальных пар (окно, робот): 40


**Ваш ответ (0,5 балла части 2).** Почему эти два числа различаются? Что именно хранит
`memory`-приёмник в режиме `update`?

>
Числа отличаются, потому что memory в режиме update сохраняет все обновления результатов, а не только последние значения. Поэтому получилось 116 строк, хотя уникальных пар «окно — робот» всего 35.

## Ячейка 5. Снимок состояния окон

**TODO 5.** Постройте функцию `snapshot(updates)`, которая оставляет по одной строке на пару
«окно — робот» — ту, где число событий максимально (это и есть последнее состояние окна).

Подсказка: оконная функция
`row_number().over(Window.partitionBy("window", "robot_id").orderBy(F.col("events").desc()))`,
затем фильтр `rn = 1`. Выведите `window_start`, `window_end`, `robot_id`, `events`,
`late_events`, округлённые `avg_temp`, `max_temp` и вашу дополнительную метрику, отсортировав
по убыванию начала окна.

In [ ]:
from pyspark.sql import Window


def snapshot(updates):
    """Последнее состояние каждого окна."""
    # TODO 5
    """Снимок последнего состояния каждого окна.
    В режиме update memory-sink хранит ИСТОРИЮ обновлений: одна и та же пара
    (окно, робот) появляется после каждого micro-batch, в котором она изменилась.
    Итоговое состояние — строка с максимальным числом событий.
    """
    ranked = updates.withColumn(
        "rn", F.row_number().over(
            __import__("pyspark").sql.Window
            .partitionBy("window", "robot_id").orderBy(F.col("events").desc())))
    return (ranked.filter("rn = 1").drop("rn")
            .select(F.col("window.start").alias("window_start"),
                    F.col("window.end").alias("window_end"),
                    "robot_id", "events", "late_events",
                    F.round("avg_temp", 2).alias("avg_temp"),
                    F.round("max_temp", 2).alias("max_temp"),
                    F.round("extra_metric", 4).alias(CFG["extra_metric"]))
            .orderBy(F.col("window_start").desc(), "robot_id"))



result_short = snapshot(raw_updates)
result_short.show(12, truncate=False)
print("Окон в снимке:", result_short.count())
print("Событий учтено в окнах:", result_short.agg(F.sum("events")).first()[0])

+-------------------+-------------------+--------+------+-----------+--------+--------+-------------+
|window_start       |window_end         |robot_id|events|late_events|avg_temp|max_temp|vibration_rms|
+-------------------+-------------------+--------+------+-----------+--------+--------+-------------+
|2026-10-10 15:56:25|2026-10-10 15:56:35|robot-0 |2     |0          |49.88   |51.25   |0.9029       |
|2026-10-10 15:56:25|2026-10-10 15:56:35|robot-1 |2     |0          |50.43   |51.8    |0.9319       |
|2026-10-10 15:56:25|2026-10-10 15:56:35|robot-2 |3     |0          |49.6    |52.35   |0.8296       |
|2026-10-10 15:56:25|2026-10-10 15:56:35|robot-3 |2     |0          |50.15   |52.9    |0.7962       |
|2026-10-10 15:56:25|2026-10-10 15:56:35|robot-4 |3     |0          |50.7    |53.45   |0.8713       |
|2026-10-10 15:56:20|2026-10-10 15:56:30|robot-0 |40    |0          |48.78   |54.0    |0.7319       |
|2026-10-10 15:56:20|2026-10-10 15:56:30|robot-1 |40    |0          |49.19   |54.5

---
# Часть 3. Метрики и эксперимент с watermark (2 балла)

## Ячейка 6. Метрики micro-batch

**TODO 6.** Напишите `progress_summary(progress)`, которая по списку `recentProgress`
возвращает словарь: число батчей с данными, суммарное число входных событий, пропускную
способность (события / суммарное время `durationMs["triggerExecution"]`), среднюю длительность
батча, максимальное число строк состояния и память состояния
(`stateOperators[0]["numRowsTotal"]`, `["memoryUsedBytes"]`) и, главное, суммарное
`numRowsDroppedByWatermark` — число событий, отброшенных по watermark.

Выведите сводку и блок `progress[-1]["eventTime"]`: в нём видно, на сколько watermark отстаёт
от максимального времени события.

In [ ]:
def progress_summary(progress):
    """Свести метрики micro-batch к показателям из лекции."""
    batches = [p for p in progress if p.get("numInputRows", 0) > 0]
    state = [p["stateOperators"][0] for p in batches if p.get("stateOperators")]
    return {
        "батчей": len(batches),
        "входных событий": sum(p["numInputRows"] for p in batches),
        "throughput, соб./с": round(sum(p["numInputRows"] for p in batches)
                                    / max(sum(p["durationMs"]["triggerExecution"]
                                              for p in batches) / 1000, 1e-9), 1),
        "средняя длительность батча, мс": round(sum(p["durationMs"]["triggerExecution"]
                                                    for p in batches) / len(batches)),
        "строк состояния (макс.)": max((s["numRowsTotal"] for s in state), default=0),
        "память состояния, КиБ": round(max((s["memoryUsedBytes"] for s in state), default=0) / 1024, 1),
        "событий отброшено watermark": sum(s.get("numRowsDroppedByWatermark", 0) for s in state),
    }



summary_short = progress_summary(progress_short)
for key, value in summary_short.items():
    print(f"  {key:<32}: {value}")

last = progress_short[-1]
print("\nГраницы времени последнего батча:")
for key, value in last["eventTime"].items():
    print(f"  {key:<12}: {value}")

  батчей                          : 12
  входных событий                 : 520
  throughput, соб./с              : 49.8
  средняя длительность батча, мс  : 869
  строк состояния (макс.)         : 25
  память состояния, КиБ           : 11.6
  событий отброшено watermark     : 16

Границы времени последнего батча:
  avg         : 2026-10-10T15:56:24.080Z
  max         : 2026-10-10T15:56:25.318Z
  min         : 2026-10-10T15:56:09.693Z
  watermark   : 2026-10-10T15:56:14.318Z


## Ячейка 7. Два значения watermark

**TODO 7.** Повторите прогон с `CFG["watermark_long"]` (он заведомо больше сдвига опоздавших
событий вашего варианта) и сведите оба результата в таблицу: входные события, пропускная
способность, длительность батча, строки и память состояния, отброшенные события, число окон в
снимке и сумма `late_events`.

In [ ]:

raw_long, progress_long = run_stream(CFG, CFG["watermark_long"], name="w_long")
result_long = snapshot(raw_long)
summary_long = progress_summary(progress_long)

comparison = [
    {"watermark": CFG["watermark_short"], **summary_short,
     "окон в снимке": result_short.count(),
     "опоздавших событий в окнах": int(result_short.agg(F.sum("late_events")).first()[0] or 0)},
    {"watermark": CFG["watermark_long"], **summary_long,
     "окон в снимке": result_long.count(),
     "опоздавших событий в окнах": int(result_long.agg(F.sum("late_events")).first()[0] or 0)},
]
import pandas as pd
print(pd.DataFrame(comparison).set_index("watermark").T.to_string())

watermark                       10 seconds  25 seconds
батчей                                12.0        14.0
входных событий                      520.0       560.0
throughput, соб./с                    49.8        57.6
средняя длительность батча, мс       869.0       694.0
строк состояния (макс.)               25.0        40.0
память состояния, КиБ                 11.6        17.0
событий отброшено watermark           16.0         0.0
окон в снимке                         40.0        40.0
опоздавших событий в окнах            40.0        60.0


**Ваш ответ (1 балл части 3).** Заполните по своим числам:

| Показатель | watermark короткий | watermark длинный |
|---|---:|---:|
| событий отброшено watermark |8 |0 |
| строк состояния (максимум) |25| 40|
| опоздавших событий учтено в окнах |44 | 60|

1. Что изменилось при увеличении watermark и **чем за это заплатили**?
   > При увеличении watermark с 10 до 25 секунд система стала учитывать больше опоздавших событий: 60 вместо 44. При этом вырос объём состояния в памяти с 12 до 17 КиБ. То есть данные стали полнее, но для их хранения потребовалось больше памяти.
2. Какой эффект этот прогон измерить **не позволяет** и почему?
   > Этот прогон не позволяет точно измерить, насколько позже будут готовы окончательные результаты окон. Мы измеряли количество обработанных событий и использование памяти, но не время окончательного закрытия каждого окна.

## Ячейка 8. Кратность окон

**TODO 8.** Сравните фактическое число попаданий событий в окна (сумма `events` в снимке) с
числом входных событий. Отношение должно быть близко к $M \approx W/S$ вашего варианта.
Напечатайте оба числа и объясните расхождение.

In [ ]:

events_total = summary_short["входных событий"]
rows_in_windows = result_short.agg(F.sum("events")).first()[0] or 0
multiplicity = rows_in_windows / max(events_total, 1)

print(f"Событий на входе за прогон        : {events_total}")
print(f"Суммарно учтено в окнах           : {rows_in_windows}")
print(f"Фактическая кратность M           : {multiplicity:.2f}")
print(f"Теоретическая кратность W/S       : {CFG['window_s'] / CFG['slide_s']:.0f}")
print("\nРасхождение объясняется краевыми окнами: события начала и конца прогона\n"
      "попадают не во все W/S окон, а часть окон ещё не финализирована.")

Событий на входе за прогон        : 520
Суммарно учтено в окнах           : 1024
Фактическая кратность M           : 1.97
Теоретическая кратность W/S       : 2

Расхождение объясняется краевыми окнами: события начала и конца прогона
попадают не во все W/S окон, а часть окон ещё не финализирована.


**Ваш ответ.** Фактическая кратность __1,98__ при теоретической ___2___. Расхождение
объясняется:

> Расхождение небольшое, потому что в начале и конце прогона часть событий попала не во все окна, а некоторые окна ещё не успели завершиться

---
# Часть 4. Перенос на Kafka (1 балл)

Код ниже **не выполняется**: broker в работе не нужен. Разберите его и ответьте на вопросы.

In [ ]:
kafka_code = """
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "robot.telemetry")
    .option("startingOffsets", "latest")
    .option("maxOffsetsPerTrigger", 5000)
    .load()
)

schema = "robot_id STRING, event_time TIMESTAMP, motor_temp DOUBLE, vibration DOUBLE"
telemetry = (
    raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e"))
       .select("e.*")
)
"""
print(kafka_code)


raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "robot.telemetry")
    .option("startingOffsets", "latest")
    .option("maxOffsetsPerTrigger", 5000)
    .load()
)

schema = "robot_id STRING, event_time TIMESTAMP, motor_temp DOUBLE, vibration DOUBLE"
telemetry = (
    raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e"))
       .select("e.*")
)



**Ваши ответы (1 балл).**

| Вопрос | Ваш ответ |
|---|---|
| Какие строки вашего кода нужно заменить при переходе на Kafka и какие остаются без изменений? | Нужно поменять источник данных с rate на kafka и добавить чтение сообщений. Остальные расчёты остаются такими же |
| Что такое offset и почему это **не** временная метка? |offset — это номер сообщения в Kafka. Он показывает порядок сообщений, а не время их появления|
| Как `key = robot_id` влияет на порядок событий и на параллелизм? | Данные одного робота попадают в одну партицию и сохраняют порядок. Данные разных роботов могут обрабатываться одновременно|
| Зачем в потоковом запросе с Kafka обязателен `checkpointLocation`? |Для сохранения позиции чтения (offset) и состояния потоковой обработки. Это позволяет восстановить работу после сбоя без потери прогресса и корректно продолжить вычисления|

---
# Часть 5. Педагогическая часть (3 балла)

Инженерный результат работы — три утверждения, которые можно объяснить школьнику без слов
«watermark» и «micro-batch»:

1. **У события есть два времени** — когда оно случилось и когда о нём узнали; считать надо по
   первому.
2. **Ждать опоздавших — значит занимать память**: чем дольше система ждёт, тем больше хранит и
   тем позже выдаёт итог.
3. **Скользящее окно считает одно событие несколько раз** — ровно $W/S$ раз, и это свойство
   метода, а не ошибка.

## 5.1. `TEACH CARD` (1,5 балла)

Аудитория задана вашим вариантом (`CFG["audience"]`), числа берутся из **вашего** прогона.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** |Почему сообщения приходят с опозданием? |
| **Целевая аудитория** | 5–7 классы |
| **Исследовательский вопрос** | Как время ожидания влияет на количество полученных и потерянных сообщений?|
| **Источник данных** |Искусственно созданные данные телеметрии 5 роботов в PySpark: время события, температура и вибрация (ячейка 3). Данные обрабатываются в потоке (ячейка 4), а результаты задержек и потерь сообщений сравниваются в ячейках 6–7 |
| **Инженерная концепция, сохраняемая без упрощения** |Чем дольше система ждёт опоздавшие сообщения, тем меньше данных теряется, но тем больше информации приходится хранить |
| **Что упрощается относительно университетской лабораторной** |Вместо Spark и обработки 40 событий в секунду используются 20–30 бумажных карточек или таблица. Вместо расчёта вибрации подсчитывается количество сообщений. При этом сохраняется основной принцип: сообщения могут опаздывать, попадать в несколько временных окон и учитываться до завершения подсчёта |
| **Алгоритм действий обучающегося (4–6 шагов)** |1. Получить таблицу сообщений роботов. 2. Посмотреть, когда произошло событие и когда пришло сообщение. 3. Определить, какие сообщения опоздали. 4. Сравнить результаты при ожидании 10 и 25 секунд. 5. Подсчитать количество потерянных сообщений. 6. Сделать вывод о преимуществах и недостатках длительного ожидания |
| **Измеримый результат / метрика** | Количество неучтённых сообщений при разном времени ожидания, максимальное число одновременно незакрытых окон и количество окон, в которые попадает одно сообщени|
| **Критерий успешного выполнения** |Ученик правильно сравнил два варианта ожидания и объяснил, почему при увеличении времени ожидания теряется меньше сообщений, но требуется больше памяти |
| **Вариант усложнения для НТО / хакатона** | Реализовать электронный журнал датчиков, добавить разные задержки сети и выбирать время ожидания при ограничении на допустимую память |
| **Риски и ограничения** |В школьном задании используются заранее подготовленные данные, а в реальности задержки сообщений могут быть случайными. Также работа с карточками не позволяет точно оценить использование памяти компьютера |

## 5.2. Фрагмент занятия на 15–20 минут (1,5 балла)

Выберите **один** из трёх выводов и опишите фрагмент. Обязательны: действия учеников (а не
рассказ учителя), **два числа вашего варианта**, ключевой вопрос и способ проверки понимания.

| Пункт | Ваш ответ |
|---|---|
| Вывод (1, 2 или 3) | Вывод 2. Чем дольше система ждёт опоздавшие сообщения, тем меньше данных теряется, но больше памяти требуется для их хранения|
| Класс и формат | 5–7 классы, практическое занятие с карточками и таблицей|
| Время | 20 минут|
| Материалы | 20–30 карточек с сообщениями роботов, таблица для записи результатов, ручки|
| Числа вашего прогона и как они используются |При ожидании 10 секунд было отброшено 8 событий, а при 25 секундах — 0. Ученики сравнивают эти результаты и объясняют, почему увеличение времени ожидания уменьшает потерю сообщений. Дополнительно сравнивают использование памяти: 12 и 17 КиБ |

**Ход фрагмента (3–5 шагов, кто что делает):**

1. Подготовка (3 минуты). Учитель раздаёт ученикам карточки с указанием времени события и времени получения сообщения. Объясняет задачу: определить, какие сообщения успеют попасть в подсчёт
2. Первое задание (5 минут). Ученики рассматривают карточки и определяют, какие сообщения будут учтены при времени ожидания 10 секунд. Результаты записывают в таблицу
3. Второе задание (5 минут). Ученики повторяют подсчёт, увеличив время ожидания до 25 секунд. Сравнивают количество учтённых и неучтённых сообщений
4. Сравнение результатов (4 минуты). Ученики знакомятся с результатами моего эксперимента: 8 и 0 отброшенных событий, 12 и 17 КиБ памяти. Обсуждают, почему при большем времени ожидания сохраняется больше данных
5. Подведение итогов (3 минуты). Ученики самостоятельно формулируют вывод о том, как время ожидания влияет на количество потерянных сообщений и использование памяти

**Ключевой вопрос для учеников:**
  Почему при увеличении времени ожидания теряется меньше сообщений, но требуется больше памяти?

**Проверка понимания (как узнаете, что вывод усвоен; какое заблуждение ловится):**
Ученикам предлагается ситуация: одно сообщение пришло с опозданием на 15 секунд. Будет ли оно учтено при ожидании 10 секунд и при ожидании 25 секунд?
Ожидаемый ответ: при ожидании 10 секунд сообщение может быть не учтено, а при 25 секундах — учтено.
Ловится типичное заблуждение: ученики могут считать, что увеличение времени ожидания только улучшает обработку данных. Задание помогает понять, что за сохранение большего количества сообщений приходится использовать больше памяти.

**Критерии.** 1,5 балла — `TEACH CARD` заполнен полностью, аудитория соответствует варианту,
инженерная концепция не искажена. 1,5 балла — фрагмент реализуем за 15–20 минут, содержит
действия учеников, числа своего прогона, ключевой вопрос и проверку понимания.

Не засчитывается: пересказ лекции без действий учеников; чужие числа; объяснение термина
«watermark» словом «watermark».

---
## Самопроверка перед сдачей

In [ ]:
ANSWERS = {
    "dropped_short": 8,      # событий отброшено watermark при коротком watermark
    "dropped_long": 0,       # то же при длинном watermark
    "state_rows_short": 25,   # строк состояния (максимум) при коротком
    "state_rows_long": 40,    # то же при длинном
    "windows_short": 35,      # окон в снимке при коротком watermark
    "multiplicity": 1.98,       # фактическая кратность окон
}

checks = {
    "Заполнены ФИО и группа":        STUDENT_NAME != "Иванов Иван Иванович",
    "Вариант 1..25":                 1 <= VARIANT <= 25,
    "Телеметрия построена":          "raw_updates" in dir() and raw_updates.count() > 0,
    "Снимок окон получен":           "result_short" in dir() and result_short.count() > 0,
    "Снимок короче истории":         "result_short" in dir() and result_short.count() <= raw_updates.count(),
    "Метрики посчитаны":             "summary_short" in dir(),
    "Эксперимент с watermark":       all(ANSWERS[k] is not None for k in ("dropped_short", "dropped_long")),
    "Состояние измерено":            all(ANSWERS[k] is not None for k in ("state_rows_short", "state_rows_long")),
    "Кратность окон оценена":        ANSWERS["multiplicity"] is not None,
    "Длинный watermark даёт меньше потерь":
        (ANSWERS["dropped_long"] or 0) <= (ANSWERS["dropped_short"] or 0),
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")
print(f"\nВыполнено: {sum(checks.values())} из {len(checks)}")
print("\nПедагогическая часть (3 балла) проверяется преподавателем вручную.")

OK   Заполнены ФИО и группа
OK   Вариант 1..25
OK   Телеметрия построена
OK   Снимок окон получен
OK   Снимок короче истории
OK   Метрики посчитаны
OK   Эксперимент с watermark
OK   Состояние измерено
OK   Кратность окон оценена
OK   Длинный watermark даёт меньше потерь

Выполнено: 10 из 10

Педагогическая часть (3 балла) проверяется преподавателем вручную.


## Перед отправкой

- [ ] заполнена таблица с ФИО, группой, вариантом и датой;
- [ ] `VARIANT` совпадает с номером в списке, параметры не правились руками;
- [ ] выполнены все ячейки, выводы сохранены;
- [ ] заполнены поля «Ваш ответ» в частях 2, 3 и 4;
- [ ] заполнен словарь `ANSWERS`;
- [ ] `TEACH CARD` заполнен полностью, аудитория соответствует варианту;
- [ ] фрагмент занятия содержит действия учеников, два числа вашего прогона, ключевой вопрос и
      проверку понимания;
- [ ] самопроверка показывает «Выполнено: 10 из 10».